# Layerwise stream mean norm over training

Part of the experiments_anim split (1: boundary mean spectra, 2: eigval spectra, 3: coupling heatmaps, 4: layerwise RankMe, 5: layerwise mean norm) — split so saved outputs stay small enough for the editor.


In [ ]:
import os, sys
import warnings
import importlib
if os.path.basename(os.getcwd()) == 'analysis':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
import utils.model_registry, utils.accessor
importlib.reload(utils.model_registry)   # deps first: reload(_lib) alone re-imports cached modules
importlib.reload(utils.accessor)
from analysis import experiments_lib as _lib
importlib.reload(_lib)
from analysis.experiments_lib import (
    build_hooks, get_ys, get_series_y, panel_palettes, smooth_spectrum,
    submatrix, block_mean_cos, model_name_options, YVAR_LABELS, XVAR_FNS)
# nanochat zero-inits c_proj: step-0 attn/mlp.out frames are all-zero; ylim is pinned
# globally so the log-autoscale warning on those frames is pure noise.
warnings.filterwarnings('ignore', message='Data has no positive values')

In [ ]:
# Config this notebook needs (kept out of the generic lib):
BLOCK_REPR = 'block_representations_all'         # all-layer cov runs (the 4 main models)
BLOCK_SAMPLES = 'block_representations_samples'  # samples runs (also carry 160m/410m)
SRC = {                                          # model -> source for spectra/means/eigvals
    'pythia-160m-deduped':  BLOCK_SAMPLES,
    'pythia-410m-deduped':  BLOCK_SAMPLES,
    'pythia-1b-deduped':    BLOCK_REPR,
    'pythia-6.9b-deduped':  BLOCK_REPR,
    'OLMo-2-0425-1B':       BLOCK_REPR,
    'OLMo-2-1124-7B':       BLOCK_REPR,
    'nanochat-d12':         'nanochat_samples',
}
SAMPLES_SRC = {m: 'nanochat_samples' if m == 'nanochat-d12' else BLOCK_SAMPLES for m in SRC}
measured_br = {m: list(range(L)) for m, L in [
    ('pythia-160m-deduped', 12), ('pythia-410m-deduped', 24),
    ('pythia-1b-deduped', 16),   ('pythia-6.9b-deduped', 32),
    ('OLMo-2-0425-1B', 16),      ('OLMo-2-1124-7B', 32),
    ('nanochat-d12', 12)]}
# sequential blocks (OLMo-2, nanochat) expose a distinct mlp.in; parallel Pythia aliases attn.in
has_mlp_in = lambda model: 'olmo' in model.lower() or 'nanochat' in model.lower()
HK = build_hooks()                     # hook-name -> (leaf, metric) table
def bnd(model, prefix, ms):
    return [(SRC[model], HK[f'{prefix}{l}_{m.upper()[:2]}'], f'{m} {l} {prefix}')
            for l in measured_br[model] for m in ms]
attn_in  = lambda model, ms=['Au']: bnd(model, 'AI', ms)
attn_out = lambda model, ms=['Au']: bnd(model, 'AO', ms)
mlp_in   = lambda model, ms=['Au']: bnd(model, 'MI', ms)
mlp_out  = lambda model, ms=['Au']: bnd(model, 'MO', ms)

In [ ]:
# Animated-spectra engine (analysis/spectrum_anim.py): inject this notebook's data
# backend, expose animate_spectra. The notebook drives animations via animate_spectra /
# anim_meancov / anim_blkres (no plot_spectrum needed here).
import analysis.spectrum_anim as sa
importlib.reload(sa)   # pick up engine edits without restarting the kernel
sa.configure(get_series_y=get_series_y, get_ys=get_ys,
             panel_palettes=panel_palettes, smooth_spectrum=smooth_spectrum,
             submatrix=submatrix, block_mean_cos=block_mean_cos,
             model_name_options=model_name_options, YVAR_LABELS=YVAR_LABELS, XVAR_FNS=XVAR_FNS)
animate_spectra = sa.animate_spectra

In [ ]:
from IPython.display import display

## Layerwise stream $\|\mu\|_2$ over training (all models)

Each frame is the depth profile of the residual-stream mean norm $\|\mu\|_2$
(uncentered acts, `attn.in` per block plus `before_final_norm` and `after_final_norm`).


In [ ]:
def anim_layer_mean_norm(model, save_dir=None, **kw):
    src = SAMPLES_SRC[model]                      # depth profile needs the all-layer samples runs
    layers = [(src, (f'blk{k}.attn.in', 'acts'), f'blk{k}') for k in measured_br[model]]
    layers += [(src, ('before_final_norm', 'acts'), 'bfn'),
               (src, ('after_final_norm', 'acts'), 'afn')]
    panels = [('mean_norm', layers, [model], {'kind': 'profile', 'ylog': True,
               'title': r'stream $\|\mu\|_2$ by depth'})]
    save = f'{save_dir}/layer_mean_norm_{model}.mp4' if save_dir else None
    display(animate_spectra(panels, ncols=1, save=save, model=model,
                            suptitle=f'Layerwise stream mean norm — {model}', **kw))


In [ ]:
anim_layer_mean_norm('pythia-160m-deduped', save_dir='analysis/figures/animations')

In [ ]:
anim_layer_mean_norm('pythia-410m-deduped', save_dir='analysis/figures/animations')

In [ ]:
anim_layer_mean_norm('pythia-1b-deduped', save_dir='analysis/figures/animations')

In [ ]:
anim_layer_mean_norm('pythia-6.9b-deduped', save_dir='analysis/figures/animations')

In [ ]:
anim_layer_mean_norm('OLMo-2-0425-1B', save_dir='analysis/figures/animations')

In [ ]:
anim_layer_mean_norm('OLMo-2-1124-7B', save_dir='analysis/figures/animations')

In [ ]:
anim_layer_mean_norm('nanochat-d12', save_dir='analysis/figures/animations')